# Day 1 — Iris Flower Classification

The Iris dataset is a classic. 150 flowers, three species, four measurements each.  
The goal is to predict species from sepal/petal length and width.

It's been done a thousand times, but doing it yourself end-to-end is the point.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

%matplotlib inline
sns.set_style('darkgrid')

## 1. Load & Inspect

In [ ]:
iris = load_iris()

df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = iris.target
df['species_name'] = df['species'].map({
    0: 'Iris-setosa',
    1: 'Iris-versicolor',
    2: 'Iris-virginica'
})

print('Shape:', df.shape)
df.head()

In [ ]:
df.describe().round(2)

In [ ]:
# balanced classes — no need to worry about imbalance here
df['species_name'].value_counts()

## 2. EDA

In [ ]:
# distribution of each feature per species
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Feature Distributions by Species')

colors = {'Iris-setosa': '#4CAF50', 'Iris-versicolor': '#2196F3', 'Iris-virginica': '#FF5722'}

for ax, feature in zip(axes.flatten(), iris.feature_names):
    for species, color in colors.items():
        subset = df[df['species_name'] == species]
        ax.hist(subset[feature], alpha=0.65, label=species, color=color, bins=15)
    ax.set_title(feature)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# petal length alone separates setosa perfectly
# versicolor and virginica overlap — that's where the model earns its keep

plt.figure(figsize=(8, 5))
for species, color in colors.items():
    subset = df[df['species_name'] == species]
    plt.scatter(
        subset['petal length (cm)'],
        subset['petal width (cm)'],
        c=color, label=species, alpha=0.75, s=50
    )
plt.xlabel('Petal Length (cm)')
plt.ylabel('Petal Width (cm)')
plt.title('Petal Length vs Petal Width')
plt.legend()
plt.show()

In [ ]:
# correlation heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(
    df[iris.feature_names].corr(),
    annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5
)
plt.title('Feature Correlation')
plt.show()

# petal length and petal width are strongly correlated (0.96)
# that's expected — bigger flowers are bigger everywhere

## 3. Train / Test Split

In [ ]:
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y      # keeps class ratios the same in train/test
)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # fit only on train — important

## 4. Model Training

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train_scaled, y_train)

cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
print(f'CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 5. Evaluation

In [ ]:
y_pred = model.predict(X_test_scaled)
print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.4f}\n')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
importances = model.feature_importances_
feat_names  = [f.replace(' (cm)', '') for f in iris.feature_names]
sorted_idx  = np.argsort(importances)[::-1]

plt.figure(figsize=(7, 4))
plt.bar([feat_names[i] for i in sorted_idx], importances[sorted_idx],
        color=['#4CAF50', '#2196F3', '#FF5722', '#9C27B0'])
plt.title('Feature Importance')
plt.ylabel('Importance Score')
plt.tight_layout()
plt.show()

# petal measurements do most of the work — not surprising given the scatter plot above

## 6. Single Prediction Example

In [ ]:
sample = np.array([[5.1, 3.5, 1.4, 0.2]])   # typical setosa
sample_scaled = scaler.transform(sample)

pred  = model.predict(sample_scaled)[0]
proba = model.predict_proba(sample_scaled)[0]

print('Prediction:', iris.target_names[pred])
print('Probabilities:')
for name, p in zip(iris.target_names, proba):
    print(f'  {name}: {p:.2%}')

## Wrap-up

**Accuracy: 90% on test set, ~95% on 5-fold CV.**

The 3 misclassifications are all between versicolor and virginica — which makes sense. Those two overlap heavily in feature space. Setosa is easy; the model gets every single setosa right.

